In [17]:
import pandas as pd
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)

In [18]:
train_df = pd.read_parquet("train.parquet")
val_df = pd.read_parquet("validation.parquet")


langs = ["te", "ar", "ko"]

train_df = train_df[train_df["lang"].isin(langs)]
val_df = val_df[val_df["lang"].isin(langs)]
train_df_ko = train_df[train_df["lang"] == "ko"]
train_df_ar = train_df[train_df["lang"] == "ar"]
train_df_te = train_df[train_df["lang"] == "te"]
val_df_ko = val_df[val_df["lang"] == "ko"]
val_df_ar = val_df[val_df["lang"] == "ar"]
val_df_te = val_df[val_df["lang"] == "te"]

print(len(train_df))
train_df

6335


,question,context,lang,answerable,answer_start,answer,answer_inlang
4792,30년 전쟁의 승자는 누구인가?,The conflict between France and Spain continue...,ko,True,21,France,None
4793,엑스선은 누가 발견하였는가?,"X-rays make up X-radiation, a form of electrom...",ko,True,503,Wilhelm Röntgen,None
4794,아테네에서 언제 가장 최근의 올림픽이 올렸나요?,"In 2022, Beijing will become the first-ever ci...",ko,True,188,2004,None
4795,세상에서 가장 오래된 방송사는 무엇인가?,The British Broadcasting Corporation (BBC) is ...,ko,True,4,British Broadcasting Corporation (BBC),None
4796,팔레스타인 수도는 어딘가요?,"Palestine ( '), officially the State of Palest...",ko,True,205,Jerusalem,None
...,...,...,...,...,...,...,...
15338,소말리아는 2차 개헌을 언제 했나요?,"In February 2012, Somali government officials ...",ko,True,923,23 June 2012,None
15339,세상에서 가장 먼저 시작된 교통수단은 무엇인가?,The first earth tracks were created by humans ...,ko,True,160,animals,None
15340,2019년 이집트의 지도자는 누구인가?,"Abdel Fattah Saeed Hussein Khalil El-Sisi ( """"...",ko,True,0,Abdel Fattah Saeed Hussein Khalil El-Sisi,None
15341,독일에서 가장 인구밀도가 높은 도시는 무엇인가?,Munich (; ; ) is the capital and most populous...,ko,True,205,Berlin,None


In [19]:
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)

# Initialize
tokenizer = AutoTokenizer.from_pretrained('xlm-roberta-base')
model = AutoModelForTokenClassification.from_pretrained(
    'xlm-roberta-base',
    num_labels=3,
    id2label={0: 'O', 1: 'B-ANS', 2: 'I-ANS'},
    label2id={'O': 0, 'B-ANS': 1, 'I-ANS': 2}
)


Some weights of XLMRobertaForTokenClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [20]:
def preprocess_function(examples):
    # Tokenize
    tokenized = tokenizer(
        examples['question'],
        examples['context'],
        truncation='only_second',
        max_length=512,
        padding='max_length',
        return_offsets_mapping=True
    )

    labels_batch = []

    for i in range(len(examples['question'])):
        labels = [-100] * len(tokenized['input_ids'][i])

        # If answerable, mark answer span
        if examples['answerable'][i] and examples['answer_start'][i] is not None:
            answer_start = int(examples['answer_start'][i])
            answer_end = answer_start + len(examples['answer'][i])

            # Map to tokens
            offset_mapping = tokenized['offset_mapping'][i]
            first_token = True

            for idx, (start, end) in enumerate(offset_mapping):
                if start == 0 and end == 0:  # Special token
                    continue

                # Check if token overlaps with answer
                if start >= answer_start and end <= answer_end:
                    if first_token:
                        labels[idx] = 1  # B-ANS
                        first_token = False
                    else:
                        labels[idx] = 2  # I-ANS
                elif start < answer_end and end > answer_start:  # Partial overlap
                    if first_token:
                        labels[idx] = 1
                        first_token = False
                    else:
                        labels[idx] = 2
                elif end > answer_start:  # Past answer
                    break

        labels_batch.append(labels)

    tokenized['labels'] = labels_batch
    return tokenized


In [21]:
train_dataset = train_dataset.map(preprocess_function, batched=True)


Map:   0%|          | 0/6335 [00:00<?, ? examples/s]

In [22]:
def preprocess_val(dataset):
    dataset = dataset.map(
        preprocess_function,
        batched=True,
        remove_columns=dataset.column_names
    )
    # Remove examples with all -100 labels
    dataset = dataset.filter(lambda x: any(l != -100 for l in x["labels"]))
    return dataset

val_dataset = preprocess_val(val_dataset)


Map:   0%|          | 0/1155 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1155 [00:00<?, ? examples/s]

In [24]:


training_args = TrainingArguments(
    output_dir='./results',
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    num_train_epochs=5,
    eval_strategy='steps',
    eval_steps=250,
    load_best_model_at_end=True,
    report_to='none'
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=DataCollatorForTokenClassification(tokenizer),
)


In [25]:
trainer.train()

Step,Training Loss,Validation Loss
250,No log,0.234869
500,0.285100,0.208523
750,0.285100,0.200814
1000,0.214700,0.219797
1250,0.214700,0.200127
1500,0.181600,0.187244
1750,0.181600,0.206559
2000,0.154900,0.206539
2250,0.154900,0.205414
2500,0.133300,0.249970


TrainOutput(global_step=3960, training_loss=0.15914839253281102, metrics={'train_runtime': 4221.0325, 'train_samples_per_second': 7.504, 'train_steps_per_second': 0.938, 'total_flos': 8276649597619200.0, 'train_loss': 0.15914839253281102, 'epoch': 5.0})

In [26]:
# Save model
model.save_pretrained("./saved_model")

# Save tokenizer
tokenizer.save_pretrained("./saved_model")
from google.colab import drive
drive.mount('/content/drive')
model_path = "/content/drive/MyDrive/roberta_week40"
# Save to your Drive
model.save_pretrained(model_path)
tokenizer.save_pretrained(model_path)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


('/content/drive/MyDrive/roberta_week40/tokenizer_config.json',
 '/content/drive/MyDrive/roberta_week40/special_tokens_map.json',
 '/content/drive/MyDrive/roberta_week40/sentencepiece.bpe.model',
 '/content/drive/MyDrive/roberta_week40/added_tokens.json',
 '/content/drive/MyDrive/roberta_week40/tokenizer.json')

In [27]:
import numpy as np
from sklearn.metrics import f1_score



val_ar_dataset = Dataset.from_pandas(val_df_ar)
val_ko_dataset = Dataset.from_pandas(val_df_ko)
val_te_dataset = Dataset.from_pandas(val_df_te)


def preprocess_val(dataset):
    dataset = dataset.map(
        preprocess_function,
        batched=True,
        remove_columns=dataset.column_names
    )
    # Remove examples with all -100 labels
    dataset = dataset.filter(lambda x: any(l != -100 for l in x["labels"]))
    return dataset

val_ar_dataset = preprocess_val(val_ar_dataset)
val_ko_dataset = preprocess_val(val_ko_dataset)
val_te_dataset = preprocess_val(val_te_dataset)


def compute_token_f1(eval_dataset):
    predictions, labels, _ = trainer.predict(eval_dataset)
    preds = np.argmax(predictions, axis=-1)

    true_labels, true_preds = [], []

    for l_seq, p_seq in zip(labels, preds):
        for l, p in zip(l_seq, p_seq):
            if l != -100:  # ignore padding
                true_labels.append(l)
                true_preds.append(p)

    f1 = f1_score(true_labels, true_preds, average="macro")
    return f1




Map:   0%|          | 0/415 [00:00<?, ? examples/s]

Filter:   0%|          | 0/415 [00:00<?, ? examples/s]

Map:   0%|          | 0/356 [00:00<?, ? examples/s]

Filter:   0%|          | 0/356 [00:00<?, ? examples/s]

Map:   0%|          | 0/384 [00:00<?, ? examples/s]

Filter:   0%|          | 0/384 [00:00<?, ? examples/s]

In [29]:
f1_ar = compute_token_f1(val_ar_dataset)
f1_ko = compute_token_f1(val_ko_dataset)
f1_te = compute_token_f1(val_te_dataset)

print("Token-level F1 per language:")
print(f"Arabic: {f1_ar:.4f}")
print(f"Korean: {f1_ko:.4f}")
print(f"Telugu: {f1_te:.4f}")

Token-level F1 per language:
Arabic: 0.8543
Korean: 0.8977
Telugu: 0.8435


Start the testing

In [30]:
test_df = pd.read_json("test.json")

In [31]:
test_df

,question,context,lang,answerable,answer_start,answer,answer_inlang
0,When was the Kyivan Rus' founded?,Kyivan Rus' was a federation of Slavic tribes ...,en,True,64,late 9th century,late 9th century
1,Which city was the political center of Kyivan ...,Kyivan Rus' was a federation of Slavic tribes ...,en,True,89,Kyiv,Kyiv
2,Which prince converted Kyivan Rus' to Christia...,"In 988, Prince Volodymyr the Great adopted Chr...",en,True,9,Prince Volodymyr the Great,Prince Volodymyr the Great
3,When did Ukraine declare independence from the...,Ukraine declared its independence from the Sov...,en,True,60,24 August 1991,24 August 1991
4,Which empire ruled most of western Ukraine bef...,"Before World War I, western Ukraine was part o...",en,True,56,Austro-Hungarian Empire,Austro-Hungarian Empire
5,Was the Holodomor a man-made famine in Soviet ...,The Holodomor was a man-made famine that took ...,en,True,4,Yes,Yes
6,Did Ukraine become independent in 1989?,Ukraine declared its independence from the Sov...,en,True,60,No,No
7,Who was the first President of independent Ukr...,Ukraine declared its independence from the Sov...,en,False,-1,no answer,None
8,Did the Cossack Hetmanate sign a treaty with t...,The Pereyaslav Agreement of 1654 was a treaty ...,en,True,44,Yes,Yes
9,Who was the leader of the Ukrainian Insurgent ...,"During World War II, Ukraine became a major ba...",en,False,-1,no answer,None


In [33]:
test_dataset = Dataset.from_pandas(test_df)
test_dataset = preprocess_val(test_dataset)

Map:   0%|          | 0/32 [00:00<?, ? examples/s]

Filter:   0%|          | 0/32 [00:00<?, ? examples/s]

In [40]:
f1_test = compute_token_f1(test_dataset)
print(f"Token-level F1 on test set: {f1_test:.4f}")

Token-level F1 on test set: 0.6263


In [7]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [38]:
model_path = "/content/drive/MyDrive/roberta_week40"
model = AutoModelForTokenClassification.from_pretrained(model_path, local_files_only=True)

tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True)

